# VinDr-CXR → YOLOv8 전처리 파이프라인

PNG 1024x1024 경량 버전 (~3.6GB) 사용. 전체 한 번에 실행.

**Cell 1**: 설정 + 패키지 설치  
**Cell 2**: Kaggle 다운로드 (PNG + CSV)  
**Cell 3**: WBF 병합 + YOLO 변환 + S3 업로드  

In [ ]:
# ============================================================
# Cell 1: 설정 + 패키지 설치
# ============================================================
!pip install -q kaggle ensemble-boxes

# Kaggle 토큰 S3에서 복사
!mkdir -p ~/.kaggle
!aws s3 cp s3://pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an/config/kaggle.json ~/.kaggle/ --region ap-northeast-2
!chmod 600 ~/.kaggle/kaggle.json

print('설정 완료!')

In [ ]:
# ============================================================
# Cell 2: 데이터 로드 (로컬 → S3 → Kaggle 순서)
# ============================================================
import os, time, glob, zipfile, subprocess, shutil

BASE_DIR = '/home/ec2-user/SageMaker/vindr-cxr'
PNG_DIR = f'{BASE_DIR}/png_1024'
ANNO_DIR = f'{BASE_DIR}/annotations'
BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
S3_RAW = f's3://{BUCKET}/vindr-cxr/raw'
REGION = 'ap-northeast-2'

os.makedirs(PNG_DIR, exist_ok=True)
os.makedirs(ANNO_DIR, exist_ok=True)

def s3_exists(s3_path):
    r = subprocess.run(f'aws s3 ls {s3_path} --region {REGION}',
                       shell=True, capture_output=True, text=True)
    return r.returncode == 0 and len(r.stdout.strip()) > 0

def find_pngs():
    for d in [f'{PNG_DIR}/train', PNG_DIR, f'{BASE_DIR}/train']:
        if os.path.isdir(d) and len(glob.glob(f'{d}/*.png')) > 100:
            return d
    for root, dirs, files in os.walk(PNG_DIR):
        if sum(1 for f in files if f.endswith('.png')) > 100:
            return root
    return None

def find_csv():
    """로컬 전체에서 train.csv 찾기"""
    # 예상 경로 먼저
    candidates = [
        f'{ANNO_DIR}/train.csv',
        f'{PNG_DIR}/train.csv',
        f'{BASE_DIR}/train.csv',
        '/home/ec2-user/SageMaker/train.csv',  # 기본 업로드 경로
    ]
    for f in candidates:
        if os.path.exists(f):
            return f
    # 재귀 검색
    for p in glob.glob('/home/ec2-user/SageMaker/**/train.csv', recursive=True):
        return p
    return None

start = time.time()

# ==================== PNG 이미지 ====================
print('=' * 50)
print('[1/2] PNG 1024x1024 이미지')
print('=' * 50)

train_dir = find_pngs()
if train_dir:
    cnt = len(glob.glob(f'{train_dir}/*.png'))
    print(f'  로컬 존재: {cnt:,}장 → {train_dir}')
elif s3_exists(f'{S3_RAW}/png_1024/'):
    print(f'  S3에서 다운로드...')
    !aws s3 sync {S3_RAW}/png_1024/ {PNG_DIR}/train/ --region {REGION} --only-show-errors
    train_dir = find_pngs()
    print(f'  완료: {len(glob.glob(f"{train_dir}/*.png")):,}장')
else:
    print(f'  Kaggle 다운로드...')
    !kaggle datasets download -d xhlulu/vinbigdata-chest-xray-resized-png-1024x1024 -p {PNG_DIR}
    for zf in glob.glob(f'{PNG_DIR}/*.zip'):
        !cd {PNG_DIR} && unzip -q -o "{zf}"
        os.remove(zf)
    train_dir = find_pngs()
    cnt = len(glob.glob(f'{train_dir}/*.png')) if train_dir else 0
    print(f'  완료: {cnt:,}장')
    if train_dir:
        print(f'  S3 백업 중...')
        !aws s3 sync {train_dir}/ {S3_RAW}/png_1024/ --region {REGION} --only-show-errors
        print(f'  S3 백업 완료!')

# ==================== train.csv ====================
print(f'\n[2/2] train.csv (어노테이션)')

train_csv = find_csv()

if train_csv:
    # 찾았으면 → 정위치로 복사 + S3 백업 (자동)
    target = f'{ANNO_DIR}/train.csv'
    if train_csv != target:
        shutil.copy2(train_csv, target)
        print(f'  발견: {train_csv} → {target} 복사 완료')
        train_csv = target
    else:
        print(f'  로컬 존재: {train_csv}')
    # S3 백업
    if not s3_exists(f'{S3_RAW}/train.csv'):
        print(f'  S3 백업 중...')
        !aws s3 cp {train_csv} {S3_RAW}/train.csv --region {REGION}
        print(f'  S3 백업 완료!')
elif s3_exists(f'{S3_RAW}/train.csv'):
    print(f'  S3에서 다운로드...')
    train_csv = f'{ANNO_DIR}/train.csv'
    !aws s3 cp {S3_RAW}/train.csv {train_csv} --region {REGION}
    print(f'  완료: {train_csv}')
else:
    print(f'  Kaggle 다운로드...')
    !kaggle competitions download -c vinbigdata-chest-xray-abnormalities-detection -f train.csv -p {ANNO_DIR}
    for zf in glob.glob(f'{ANNO_DIR}/*.zip'):
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(ANNO_DIR)
        os.remove(zf)
    train_csv = find_csv()
    if train_csv:
        !aws s3 cp {train_csv} {S3_RAW}/train.csv --region {REGION}

# ==================== 최종 확인 ====================
elapsed = time.time() - start
print(f'\n{"="*50}')
print(f'데이터 준비 완료! ({elapsed:.0f}s)')
print(f'  PNG: {len(glob.glob(f"{train_dir}/*.png")) if train_dir else 0:,}장 → {train_dir}')
print(f'  CSV: {train_csv}')
if not train_dir or not train_csv:
    print('\n*** 데이터를 찾을 수 없습니다! ***')

In [ ]:
# ============================================================
# Cell 3: WBF 병합 → YOLO 변환 → S3 업로드 (한 번에)
# ============================================================
import os, time, json, glob, subprocess
import numpy as np
import pandas as pd
from ensemble_boxes import weighted_boxes_fusion

WORK_BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
BASE_DIR = '/home/ec2-user/SageMaker/vindr-cxr'
PNG_DIR = f'{BASE_DIR}/png_1024'
ANNO_DIR = f'{BASE_DIR}/annotations'
YOLO_DIR = f'{BASE_DIR}/yolo_dataset'
NUM_CLASSES = 14

CLASS_NAMES = [
    'Aortic_enlargement', 'Atelectasis', 'Calcification', 'Cardiomegaly',
    'Consolidation', 'ILD', 'Infiltration', 'Lung_Opacity',
    'Nodule_Mass', 'Other_lesion', 'Pleural_effusion', 'Pleural_thickening',
    'Pneumothorax', 'Pulmonary_fibrosis',
]

# Cell 2에서 설정한 변수 사용, 없으면 다시 검색
if 'train_dir' not in dir() or train_dir is None:
    train_dir = None
    for d in [f'{PNG_DIR}/train', PNG_DIR]:
        if os.path.isdir(d) and glob.glob(f'{d}/*.png'):
            train_dir = d
            break
    if not train_dir:
        for root, dirs, files in os.walk(PNG_DIR):
            if sum(1 for f in files if f.endswith('.png')) > 100:
                train_dir = root
                break

if 'train_csv' not in dir() or train_csv is None:
    train_csv = None
    for f in [f'{ANNO_DIR}/train.csv', f'{PNG_DIR}/train.csv', f'{BASE_DIR}/train.csv']:
        if os.path.exists(f):
            train_csv = f
            break
    if not train_csv:
        for csv_path in glob.glob(f'{BASE_DIR}/**/train.csv', recursive=True):
            train_csv = csv_path
            break
    if not train_csv:
        for csv_path in glob.glob('/home/ec2-user/SageMaker/**/train.csv', recursive=True):
            train_csv = csv_path
            break

print(f'PNG 경로: {train_dir}')
print(f'CSV 경로: {train_csv}')

# 검증: 파일 못 찾으면 진단 정보 출력 후 중단
assert train_dir is not None, f"PNG 이미지 디렉토리를 찾을 수 없습니다. Cell 2를 먼저 실행하세요."
assert train_csv is not None, (
    f"train.csv를 찾을 수 없습니다. Cell 2를 먼저 실행하세요.\n"
    f"ANNO_DIR 내용: {os.listdir(ANNO_DIR) if os.path.isdir(ANNO_DIR) else 'N/A'}\n"
    f"BASE_DIR CSV: {glob.glob(BASE_DIR + '/**/*.csv', recursive=True)}"
)

# ==================== PHASE 1: WBF 병합 ====================
print(f'\n{"="*50}')
print('Phase 1: WBF 병합')
print('=' * 50)

df = pd.read_csv(train_csv)
print(f'원본: {len(df):,}행, {df["image_id"].nunique():,}개 이미지, {df["rad_id"].nunique()}명 어노테이터')

# 클래스별 통계
print(f'\n클래스별 (병합 전):')
for cid in sorted(df['class_id'].unique()):
    name = df[df['class_id'] == cid]['class_name'].iloc[0]
    count = len(df[df['class_id'] == cid])
    print(f'  [{cid:2d}] {name:<30s} {count:,}')

# No finding 분리
nf_image_ids = df[df['class_id'] == 14]['image_id'].unique()
df_abnormal = df[df['class_id'] != 14].copy()
print(f'\nNo finding: {len(nf_image_ids):,}개 / 이상 소견: {df_abnormal["image_id"].nunique():,}개')

# WBF 병합
merged_results = []
image_ids = df_abnormal['image_id'].unique()
start = time.time()

for idx, image_id in enumerate(image_ids):
    img_df = df_abnormal[df_abnormal['image_id'] == image_id]
    orig_w = max(img_df['x_max'].max() + 10, 1024)
    orig_h = max(img_df['y_max'].max() + 10, 1024)

    boxes_list, scores_list, labels_list = [], [], []
    for rad_id in img_df['rad_id'].unique():
        rad_df = img_df[img_df['rad_id'] == rad_id]
        boxes, scores, labels = [], [], []
        for _, row in rad_df.iterrows():
            cid = int(row['class_id'])
            if cid >= NUM_CLASSES:
                continue
            x1 = max(0, row['x_min'] / orig_w)
            y1 = max(0, row['y_min'] / orig_h)
            x2 = min(1, row['x_max'] / orig_w)
            y2 = min(1, row['y_max'] / orig_h)
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append([x1, y1, x2, y2])
            scores.append(1.0)
            labels.append(cid)
        if boxes:
            boxes_list.append(boxes)
            scores_list.append(scores)
            labels_list.append(labels)

    if not boxes_list:
        continue

    try:
        fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            iou_thr=0.5, skip_box_thr=0.33, weights=None
        )
    except Exception:
        continue

    for box, score, label in zip(fused_boxes, fused_scores, fused_labels):
        merged_results.append({
            'image_id': image_id,
            'class_id': int(label),
            'x1': float(box[0]), 'y1': float(box[1]),
            'x2': float(box[2]), 'y2': float(box[3]),
        })

    if (idx + 1) % 2000 == 0 or (idx + 1) == len(image_ids):
        elapsed = time.time() - start
        pct = (idx + 1) / len(image_ids) * 100
        print(f'  {idx+1:,}/{len(image_ids):,} ({pct:.1f}%) | {elapsed:.0f}s')

merged_df = pd.DataFrame(merged_results)
print(f'\n병합 완료: {len(df_abnormal):,} → {len(merged_df):,}행 ({merged_df["image_id"].nunique():,}개 이미지)')
print(f'\n클래스별 (병합 후):')
for cid in range(NUM_CLASSES):
    count = len(merged_df[merged_df['class_id'] == cid])
    print(f'  [{cid:2d}] {CLASS_NAMES[cid]:<25s} {count:,}')

# ==================== PHASE 2: YOLO 변환 ====================
print(f'\n{"="*50}')
print('Phase 2: YOLO 변환')
print('=' * 50)

for split in ['train', 'val']:
    os.makedirs(f'{YOLO_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{YOLO_DIR}/labels/{split}', exist_ok=True)

# train/val split (80/20)
all_imgs = sorted(merged_df['image_id'].unique())
np.random.seed(42)
np.random.shuffle(all_imgs)
split_idx = int(len(all_imgs) * 0.8)
train_set = set(all_imgs[:split_idx])
val_set = set(all_imgs[split_idx:])

# No finding 30% 배경
nf_list = list(nf_image_ids)
np.random.seed(42)
np.random.shuffle(nf_list)
nf_use = nf_list[:int(len(nf_list) * 0.3)]
nf_split = int(len(nf_use) * 0.8)
nf_train = nf_use[:nf_split]
nf_val = nf_use[nf_split:]

print(f'이상소견: train {len(train_set):,} / val {len(val_set):,}')
print(f'배경(NF): train {len(nf_train):,} / val {len(nf_val):,}')

# YOLO 라벨 생성 + 이미지 링크
stats = {'train': 0, 'val': 0, 'labels': 0, 'bg_train': 0, 'bg_val': 0, 'skip': 0}
start = time.time()

for image_id, group in merged_df.groupby('image_id'):
    split = 'train' if image_id in train_set else ('val' if image_id in val_set else None)
    if not split:
        continue
    src = f'{train_dir}/{image_id}.png'
    if not os.path.exists(src):
        stats['skip'] += 1
        continue

    dst = f'{YOLO_DIR}/images/{split}/{image_id}.png'
    if not os.path.exists(dst):
        try:
            os.link(src, dst)
        except OSError:
            os.symlink(src, dst)

    with open(f'{YOLO_DIR}/labels/{split}/{image_id}.txt', 'w') as f:
        for _, row in group.iterrows():
            cid = int(row['class_id'])
            xc = max(0, min(1, (row['x1'] + row['x2']) / 2))
            yc = max(0, min(1, (row['y1'] + row['y2']) / 2))
            w = max(0.001, min(1, row['x2'] - row['x1']))
            h = max(0.001, min(1, row['y2'] - row['y1']))
            f.write(f'{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n')
            stats['labels'] += 1
    stats[split] += 1

# 배경 이미지 (빈 라벨)
for img_list, split, key in [(nf_train, 'train', 'bg_train'), (nf_val, 'val', 'bg_val')]:
    for img_id in img_list:
        src = f'{train_dir}/{img_id}.png'
        if not os.path.exists(src):
            continue
        dst = f'{YOLO_DIR}/images/{split}/{img_id}.png'
        if not os.path.exists(dst):
            try:
                os.link(src, dst)
            except OSError:
                os.symlink(src, dst)
        lbl = f'{YOLO_DIR}/labels/{split}/{img_id}.txt'
        if not os.path.exists(lbl):
            open(lbl, 'w').close()
        stats[key] += 1

print(f'\nYOLO 변환 완료! ({time.time()-start:.0f}s)')
print(f'  Train: {stats["train"]:,} + 배경 {stats["bg_train"]:,} = {stats["train"]+stats["bg_train"]:,}')
print(f'  Val:   {stats["val"]:,} + 배경 {stats["bg_val"]:,} = {stats["val"]+stats["bg_val"]:,}')
print(f'  라벨:  {stats["labels"]:,}개')
if stats['skip']:
    print(f'  스킵:  {stats["skip"]}개 (PNG 없음)')

# data.yaml 생성
yaml_txt = f"""path: /opt/ml/input/data/training
train: images/train
val: images/val

nc: {NUM_CLASSES}
names:
"""
for i, name in enumerate(CLASS_NAMES):
    yaml_txt += f'  {i}: {name}\n'
with open(f'{YOLO_DIR}/data.yaml', 'w') as f:
    f.write(yaml_txt)

# 검증
print(f'\n검증:')
for split in ['train', 'val']:
    imgs = set(os.path.splitext(f)[0] for f in os.listdir(f'{YOLO_DIR}/images/{split}'))
    lbls = set(os.path.splitext(f)[0] for f in os.listdir(f'{YOLO_DIR}/labels/{split}'))
    print(f'  {split}: 이미지 {len(imgs):,} / 라벨 {len(lbls):,} | 불일치: {len(imgs-lbls)}')

# ==================== PHASE 3: S3 업로드 ====================
print(f'\n{"="*50}')
print('Phase 3: S3 업로드')
print('=' * 50)

s3_uri = f's3://{WORK_BUCKET}/vindr-cxr/processed/'
print(f'{YOLO_DIR} → {s3_uri}')

start = time.time()
!aws s3 sync {YOLO_DIR} {s3_uri} --region ap-northeast-2 --only-show-errors
print(f'S3 업로드 완료! ({(time.time()-start)/60:.1f}분)')

# 병합 CSV도 업로드
merged_csv_path = f'{ANNO_DIR}/train_merged_wbf.csv'
merged_df.to_csv(merged_csv_path, index=False)
!aws s3 cp {merged_csv_path} s3://{WORK_BUCKET}/vindr-cxr/annotations/train_merged_wbf.csv --region ap-northeast-2 --only-show-errors

print(f'\n{"="*50}')
print('전체 완료!')
print(f'  S3 경로: {s3_uri}')
print(f'  Train: {stats["train"]+stats["bg_train"]:,}장')
print(f'  Val: {stats["val"]+stats["bg_val"]:,}장')
print(f'  라벨: {stats["labels"]:,}개')
print(f'  클래스: {NUM_CLASSES}개')
print(f'\n다음 단계: YOLOv8 학습 코드 작성 → SageMaker 학습 잡 실행')